In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
from google.colab import files

uploaded = files.upload()

Saving laptop_performance.csv to laptop_performance.csv


In [9]:


import os
import zipfile
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

from xgboost import XGBClassifier



print("==========================================")
print("1. CREATING DATASET")
print("==========================================")

np.random.seed(42)

n = 5000

data = pd.DataFrame({
    "ram_gb": np.random.choice(
        [4, 8, 12, 16, 24, 32], n
    ),

    "cpu_usage_percent": np.random.uniform(
        10, 100, n
    ),

    "ram_usage_percent": np.random.uniform(
        20, 100, n
    ),

    "disk_usage_percent": np.random.uniform(
        10, 100, n
    ),

    "cpu_temperature": np.random.uniform(
        40, 100, n
    ),

    "gpu_usage_percent": np.random.uniform(
        0, 100, n
    ),

    "battery_health_percent": np.random.uniform(
        50, 100, n
    ),

    "storage_capacity_gb": np.random.choice(
        [256, 512, 1024, 2048], n
    ),

    "free_storage_gb": np.random.uniform(
        10, 1000, n
    ),

    "laptop_age_years": np.random.uniform(
        0.5, 7, n
    ),

    "daily_usage_hours": np.random.uniform(
        1, 16, n
    ),

    "running_processes": np.random.randint(
        30, 250, n
    )
})



data["free_storage_gb"] = np.minimum(
    data["free_storage_gb"],
    data["storage_capacity_gb"] * 0.95
)



risk_score = (

    data["cpu_usage_percent"] * 0.20 +

    data["ram_usage_percent"] * 0.15 +

    data["disk_usage_percent"] * 0.10 +

    data["cpu_temperature"] * 0.15 +

    data["gpu_usage_percent"] * 0.05 +

    (100 - data["battery_health_percent"]) * 0.05 +

    data["laptop_age_years"] * 5 +

    data["daily_usage_hours"] * 1.5 +

    data["running_processes"] * 0.05
)


risk_score += np.where(
    data["free_storage_gb"] < 30,
    15,
    0
)


data["performance_risk"] = pd.cut(
    risk_score,
    bins=[-np.inf, 45, 65, np.inf],
    labels=[
        "Normal",
        "Moderate",
        "High"
    ]
)


print("Dataset created.")
print("Rows:", data.shape[0])
print("Columns:", data.shape[1])



data.to_csv(
    "laptop_performance.csv",
    index=False
)

print("Dataset saved as laptop_performance.csv")



print("\n==========================================")
print("2. DATA PREPROCESSING")
print("==========================================")


df = data.copy()


print("\nMissing values:")

print(
    df.isnull().sum().sum()
)

duplicate_count = df.duplicated().sum()

print("Duplicate rows:", duplicate_count)

if duplicate_count > 0:

    df = df.drop_duplicates()


X = df.drop(
    columns=["performance_risk"]
)

y = df["performance_risk"]


print("\nFeature count:", X.shape[1])


label_encoder = LabelEncoder()

y_encoded = label_encoder.fit_transform(y)


print("\nTarget encoding:")

for original, encoded in zip(
    label_encoder.classes_,
    label_encoder.transform(
        label_encoder.classes_
    )
):

    print(
        original,
        "->",
        encoded
    )



X_train, X_test, y_train, y_test = train_test_split(

    X,
    y_encoded,

    test_size=0.20,

    random_state=42,

    stratify=y_encoded
)


print("\nTraining samples:", X_train.shape[0])
print("Testing samples:", X_test.shape[0])



scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(
    X_train
)

X_test_scaled = scaler.transform(
    X_test
)


print("Feature scaling completed.")



print("\n==========================================")
print("3. MODEL TRAINING")
print("==========================================")


models = {

    "Logistic Regression":
        LogisticRegression(
            max_iter=1000,
            random_state=42
        ),

    "Random Forest":
        RandomForestClassifier(
            n_estimators=200,
            random_state=42
        ),

    "XGBoost":
        XGBClassifier(
            n_estimators=200,
            max_depth=6,
            learning_rate=0.1,
            random_state=42,
            eval_metric="mlogloss"
        )
}


results = []

trained_models = {}



for model_name, model in models.items():

    print(
        "\nTraining:",
        model_name
    )

    model.fit(
        X_train_scaled,
        y_train
    )

    predictions = model.predict(
        X_test_scaled
    )


    accuracy = accuracy_score(
        y_test,
        predictions
    )

    precision = precision_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    recall = recall_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        predictions,
        average="weighted",
        zero_division=0
    )


    results.append({

        "Model": model_name,

        "Accuracy": accuracy,

        "Precision": precision,

        "Recall": recall,

        "F1 Score": f1

    })


    trained_models[
        model_name
    ] = model


    print(
        "Accuracy:",
        round(accuracy, 4)
    )

    print(
        "F1 Score:",
        round(f1, 4)
    )


results_df = pd.DataFrame(
    results
)


print("\n==========================================")
print("MODEL COMPARISON")
print("==========================================")

display(
    results_df.style.format({
        "Accuracy": "{:.4f}",
        "Precision": "{:.4f}",
        "Recall": "{:.4f}",
        "F1 Score": "{:.4f}"
    })
)



best_model_name = results_df.loc[
    results_df["F1 Score"].idxmax(),
    "Model"
]


best_model = trained_models[
    best_model_name
]


print("\n==========================================")
print("BEST MODEL")
print("==========================================")

print(
    "Selected model:",
    best_model_name
)



print("\n==========================================")
print("4. CREATING FLASK PROJECT")
print("==========================================")


project = "Laptop-Performance-Predictor"


import shutil

if os.path.exists(project):

    shutil.rmtree(project)


# Create folders
os.makedirs(
    f"{project}/model"
)

os.makedirs(
    f"{project}/templates"
)

os.makedirs(
    f"{project}/static"
)

os.makedirs(
    f"{project}/dataset"
)



data.to_csv(
    f"{project}/dataset/laptop_performance.csv",
    index=False
)



joblib.dump(
    best_model,
    f"{project}/model/laptop_performance_model.joblib"
)


joblib.dump(
    scaler,
    f"{project}/model/scaler.joblib"
)


joblib.dump(
    label_encoder,
    f"{project}/model/label_encoder.joblib"
)


print("Model files saved.")



app_code = '''
from flask import Flask, render_template, request, jsonify
import pandas as pd
import joblib
import os

app = Flask(__name__)

BASE_DIR = os.path.dirname(
    os.path.abspath(__file__)
)

MODEL_PATH = os.path.join(
    BASE_DIR,
    "model",
    "laptop_performance_model.joblib"
)

SCALER_PATH = os.path.join(
    BASE_DIR,
    "model",
    "scaler.joblib"
)

ENCODER_PATH = os.path.join(
    BASE_DIR,
    "model",
    "label_encoder.joblib"
)


# Load trained ML files
model = joblib.load(
    MODEL_PATH
)

scaler = joblib.load(
    SCALER_PATH
)

label_encoder = joblib.load(
    ENCODER_PATH
)


FEATURES = [

    "ram_gb",

    "cpu_usage_percent",

    "ram_usage_percent",

    "disk_usage_percent",

    "cpu_temperature",

    "gpu_usage_percent",

    "battery_health_percent",

    "storage_capacity_gb",

    "free_storage_gb",

    "laptop_age_years",

    "daily_usage_hours",

    "running_processes"

]


@app.route("/")
def home():

    return render_template(
        "index.html"
    )


@app.route(
    "/predict",
    methods=["POST"]
)
def predict():

    try:

        data = request.get_json()

        if not data:

            return jsonify({
                "error":
                "No input data received."
            }), 400


        values = []

        for feature in FEATURES:

            if feature not in data:

                return jsonify({
                    "error":
                    f"Missing input: {feature}"
                }), 400

            values.append(
                float(data[feature])
            )


        input_df = pd.DataFrame(
            [values],
            columns=FEATURES
        )


        # Apply same scaling
        scaled_input = scaler.transform(
            input_df
        )


        # Prediction
        prediction = model.predict(
            scaled_input
        )


        # Convert encoded value
        predicted_class = (
            label_encoder
            .inverse_transform(
                prediction
            )[0]
        )


        # Confidence
        confidence = None

        if hasattr(
            model,
            "predict_proba"
        ):

            probabilities = (
                model.predict_proba(
                    scaled_input
                )[0]
            )

            confidence = round(
                float(
                    max(probabilities)
                ) * 100,
                2
            )


        return jsonify({

            "prediction":
                predicted_class,

            "confidence":
                confidence

        })


    except Exception as e:

        return jsonify({

            "error":
                str(e)

        }), 500


if __name__ == "__main__":

    app.run(
        host="127.0.0.1",
        port=5000,
        debug=True
    )
'''


with open(
    f"{project}/app.py",
    "w"
) as f:

    f.write(app_code)



html_code = '''
<!DOCTYPE html>

<html>

<head>

    <title>
        Laptop Performance Predictor
    </title>

    <link rel="stylesheet"
          href="{{ url_for(
              'static',
              filename='style.css'
          ) }}">

</head>


<body>

<div class="container">

<h1>
Laptop Performance Degradation Predictor
</h1>

<p>
Enter your laptop information
to predict performance risk.
</p>


<form id="predictionForm">


<label>RAM (GB)</label>

<input
type="number"
id="ram_gb"
value="8"
required
>


<label>CPU Usage (%)</label>

<input
type="number"
id="cpu_usage_percent"
value="50"
required
>


<label>RAM Usage (%)</label>

<input
type="number"
id="ram_usage_percent"
value="60"
required
>


<label>Disk Usage (%)</label>

<input
type="number"
id="disk_usage_percent"
value="50"
required
>


<label>CPU Temperature (°C)</label>

<input
type="number"
id="cpu_temperature"
value="65"
required
>


<label>GPU Usage (%)</label>

<input
type="number"
id="gpu_usage_percent"
value="30"
required
>


<label>Battery Health (%)</label>

<input
type="number"
id="battery_health_percent"
value="85"
required
>


<label>Storage Capacity (GB)</label>

<input
type="number"
id="storage_capacity_gb"
value="512"
required
>


<label>Free Storage (GB)</label>

<input
type="number"
id="free_storage_gb"
value="250"
required
>


<label>Laptop Age (Years)</label>

<input
type="number"
step="0.1"
id="laptop_age_years"
value="2"
required
>


<label>Daily Usage (Hours)</label>

<input
type="number"
step="0.1"
id="daily_usage_hours"
value="6"
required
>


<label>Running Processes</label>

<input
type="number"
id="running_processes"
value="80"
required
>


<button type="submit">
Predict Performance
</button>


</form>


<div id="result"></div>


</div>


<script>

document
.getElementById(
    "predictionForm"
)
.addEventListener(
    "submit",
    async function(event) {

        event.preventDefault();


        const data = {

            ram_gb:
                Number(
                    document
                    .getElementById(
                        "ram_gb"
                    ).value
                ),

            cpu_usage_percent:
                Number(
                    document
                    .getElementById(
                        "cpu_usage_percent"
                    ).value
                ),

            ram_usage_percent:
                Number(
                    document
                    .getElementById(
                        "ram_usage_percent"
                    ).value
                ),

            disk_usage_percent:
                Number(
                    document
                    .getElementById(
                        "disk_usage_percent"
                    ).value
                ),

            cpu_temperature:
                Number(
                    document
                    .getElementById(
                        "cpu_temperature"
                    ).value
                ),

            gpu_usage_percent:
                Number(
                    document
                    .getElementById(
                        "gpu_usage_percent"
                    ).value
                ),

            battery_health_percent:
                Number(
                    document
                    .getElementById(
                        "battery_health_percent"
                    ).value
                ),

            storage_capacity_gb:
                Number(
                    document
                    .getElementById(
                        "storage_capacity_gb"
                    ).value
                ),

            free_storage_gb:
                Number(
                    document
                    .getElementById(
                        "free_storage_gb"
                    ).value
                ),

            laptop_age_years:
                Number(
                    document
                    .getElementById(
                        "laptop_age_years"
                    ).value
                ),

            daily_usage_hours:
                Number(
                    document
                    .getElementById(
                        "daily_usage_hours"
                    ).value
                ),

            running_processes:
                Number(
                    document
                    .getElementById(
                        "running_processes"
                    ).value
                )

        };


        const result =
            document
            .getElementById(
                "result"
            );


        result.innerHTML =
            "Predicting...";


        try {

            const response =
                await fetch(
                    "/predict",
                    {

                        method: "POST",

                        headers: {
                            "Content-Type":
                                "application/json"
                        },

                        body:
                            JSON.stringify(
                                data
                            )

                    }
                );


            const output =
                await response.json();


            if (output.error) {

                result.innerHTML =
                    "Error: " +
                    output.error;

            }

            else {

                result.innerHTML =
                    "<h2>Prediction: " +
                    output.prediction +
                    "</h2>" +

                    "<p>Confidence: " +
                    output.confidence +
                    "%</p>";

            }


        }

        catch(error) {

            result.innerHTML =
                "Unable to connect to Flask server.";

        }

    }
);

</script>


</body>

</html>
'''


with open(
    f"{project}/templates/index.html",
    "w"
) as f:

    f.write(html_code)



css_code = '''
body {

    font-family: Arial, sans-serif;

    background: #f4f6f8;

    margin: 0;

    padding: 30px;

}


.container {

    max-width: 700px;

    margin: auto;

    background: white;

    padding: 30px;

    border-radius: 12px;

}


h1 {

    text-align: center;

}


label {

    display: block;

    margin-top: 15px;

    font-weight: bold;

}


input {

    width: 100%;

    padding: 10px;

    margin-top: 5px;

    box-sizing: border-box;

}


button {

    width: 100%;

    padding: 14px;

    margin-top: 25px;

    cursor: pointer;

}


#result {

    margin-top: 25px;

    padding: 15px;

    text-align: center;

}
'''


with open(
    f"{project}/static/style.css",
    "w"
) as f:

    f.write(css_code)



requirements = """flask
pandas
numpy
scikit-learn
joblib
xgboost
"""


with open(
    f"{project}/requirements.txt",
    "w"
) as f:

    f.write(requirements)



readme = f"""# Laptop Performance Degradation Predictor

A machine learning application that predicts laptop performance risk as:

- Normal
- Moderate
- High

## Machine Learning Models

- Logistic Regression
- Random Forest
- XGBoost

## Selected Model

{best_model_name}

## Dataset

The project uses a synthetically generated structured dataset containing 5,000 laptop records and 12 predictive features.

## Features

- RAM
- CPU usage
- RAM usage
- Disk usage
- CPU temperature
- GPU usage
- Battery health
- Storage capacity
- Free storage
- Laptop age
- Daily usage
- Running processes

## Technologies

Python
Pandas
NumPy
Scikit-learn
XGBoost
Joblib
Flask
HTML
CSS
JavaScript

## Run the Application

Install dependencies:

pip install -r requirements.txt

Run:

python app.py

Open:

http://127.0.0.1:5000

## Note

The dataset is synthetically generated for educational/project purposes and should not be interpreted as measurements collected from real laptops.
"""


with open(
    f"{project}/README.md",
    "w"
) as f:

    f.write(readme)



zip_name = (
    "Laptop-Performance-Predictor.zip"
)


with zipfile.ZipFile(
    zip_name,
    "w",
    zipfile.ZIP_DEFLATED
) as zipf:

    for root, dirs, files_list in os.walk(
        project
    ):

        for file in files_list:

            filepath = os.path.join(
                root,
                file
            )

            archive_path = os.path.relpath(
                filepath,
                project
            )

            zipf.write(
                filepath,
                archive_path
            )



print("\n==========================================")
print("PROJECT CREATED SUCCESSFULLY")
print("==========================================")

print(
    "\nSelected model:",
    best_model_name
)

print(
    "\nProject folder:",
    project
)

print(
    "\nZIP file:",
    zip_name
)


print("\nProject structure:")

for root, dirs, files_list in os.walk(
    project
):

    for file in files_list:

        print(
            os.path.join(
                root,
                file
            )
        )



from google.colab import files

files.download(
    zip_name
)

print("\nZIP download started.")
print("Download the ZIP to your Mac.")

1. CREATING DATASET
Dataset created.
Rows: 5000
Columns: 13
Dataset saved as laptop_performance.csv

2. DATA PREPROCESSING

Missing values:
0
Duplicate rows: 0

Feature count: 12

Target encoding:
High -> 0
Moderate -> 1
Normal -> 2

Training samples: 4000
Testing samples: 1000
Feature scaling completed.

3. MODEL TRAINING

Training: Logistic Regression
Accuracy: 0.984
F1 Score: 0.982

Training: Random Forest
Accuracy: 0.927
F1 Score: 0.9186

Training: XGBoost
Accuracy: 0.952
F1 Score: 0.9489

MODEL COMPARISON


,Model,Accuracy,Precision,Recall,F1 Score
0,Logistic Regression,0.9840,0.9800,0.9840,0.9820
1,Random Forest,0.9270,0.9240,0.9270,0.9186
2,XGBoost,0.9520,0.9471,0.9520,0.9489



BEST MODEL
Selected model: Logistic Regression

4. CREATING FLASK PROJECT
Model files saved.

PROJECT CREATED SUCCESSFULLY

Selected model: Logistic Regression

Project folder: Laptop-Performance-Predictor

ZIP file: Laptop-Performance-Predictor.zip

Project structure:
Laptop-Performance-Predictor/README.md
Laptop-Performance-Predictor/app.py
Laptop-Performance-Predictor/requirements.txt
Laptop-Performance-Predictor/static/style.css
Laptop-Performance-Predictor/model/label_encoder.joblib
Laptop-Performance-Predictor/model/scaler.joblib
Laptop-Performance-Predictor/model/laptop_performance_model.joblib
Laptop-Performance-Predictor/dataset/laptop_performance.csv
Laptop-Performance-Predictor/templates/index.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


ZIP download started.
Download the ZIP to your Mac.
